In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook

In [2]:
#Entry
tperiod = 8

# Specify the instance version here

In [4]:
# Specify the instance version
instance_version = "smallsize" #Versions: "original", "smallsize", "smallsize2"

# Define file names dynamically based on instance version
location_file = f"fullDist Set V Data Location-{instance_version}.xlsx"
county_file = f"fullDist Set C Data-{instance_version}.xlsx"
resource_file = f"fullDist R-{instance_version}.xlsx"
time_matrix_file = f"durationFull-{instance_version}.txt"
distance_matrix_file = f"distanceFull-{instance_version}.txt"
inputs_file = f"fullDistInputs-{instance_version}.xlsx"
k_ijt_file = f"K_ijt_data_fixed-{instance_version}.xlsx"

# Specify the number of scenarios here

In [6]:
# Define the number of scenarios
sce = list(range(10))  # Example: 10 scenarios (adjust as needed)

# Generate equal probabilities for each scenario
P = [1/len(sce)] * len(sce)

#Demand data
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate sheet names based on the number of scenarios
sheet_names = [f"Demand_{i+1}" for i in range(len(sce))]

print(f"Number of scenarios: {len(sce)}")
print(f"Generated sheet names: {sheet_names}")

Number of scenarios: 10
Generated sheet names: ['Demand_1', 'Demand_2', 'Demand_3', 'Demand_4', 'Demand_5', 'Demand_6', 'Demand_7', 'Demand_8', 'Demand_9', 'Demand_10']


In [7]:
# Location Data
df_V = pd.read_excel(location_file)
print(df_V.head())

# County Data
df_C = pd.read_excel(county_file)
print(df_C.head())

# Resource Data
df_R = pd.read_excel(resource_file)
print(df_R.head())

# Time and Distance Matrices
timeMatrix = np.loadtxt(time_matrix_file)  # Time duration matrix (seconds)
distanceMatrix = np.loadtxt(distance_matrix_file)  # Distance matrix
print(timeMatrix)
print(distanceMatrix)

# Inputs Data
df_inputs = pd.read_excel(inputs_file)
print(df_inputs.head())

# K_ijt Data
df_K = pd.read_excel(k_ijt_file)
print(df_K.head())

   Index      County      Type  Cluster  \
0      0         NaN  Supplier      NaN   
1      1  Georgetown   Shelter      9.0   
2      2       Horry   Shelter     18.0   
3      3         NaN       RSA      NaN   
4      4         NaN       RSA      NaN   

                                            Facility  \
0                    FEMA Warehouse (Food and Water)   
1  Cluster 9 (Andrews Elementary School, Andrews ...   
2                                  Loris High School   
3                                   Florence Airport   
4                              Florence Civic Center   

                                             Address   latitude  longitude  \
0  3780 Southside Industrial Pkwy, Atlanta, GA 30354 -84.373590  33.650208   
1          13072 County Line Road, Andrews, SC 29510 -79.570380  33.462943   
2               301 Loris Lion Road, Loris, SC 29569 -78.860505  34.043623   
3               2100 Terminal Dr, Florence, SC 29501 -79.728332  34.193145   
4             

In [8]:
# Load demand data
d_its = {}

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    demand_array = df.values

    for i in range(demand_array.shape[0]):
        actual_location = i  # Start location index from 0
        for t in range(demand_array.shape[1]):
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 4:", d_its[(5, 2, 0)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(6, 2, 0)])

Sample demand data for location 6, time 1, scenario 4: 127.8546055072433
Sample demand data for location 6, time 1, scenario 2: 871.116680667272


In [9]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 8
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 2
All locations: 7
Warehouse/Supplier locations: 1 [0]
Shelter locations: 2 [1, 2]
RSA locations: 2 [3, 4]
PoD locations: 2 [5, 6]
All resources: 1
Time periods: 3
Time Before Landfall: [0]
Time After Landfall: [1, 2]
Scenario indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Scenario probabilities: [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]


In [10]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[1, 5]
Number of arcs: 18
Number of arcs eliminated by this arc set: 31


In [11]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])

# Initialize inventory list
Inv = []

# Define starting inventory based on instance version
if instance_version == "original":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "smallsize":
    for i in range(V):
        Inv.append(df_V['Water StartInv'][i])
    I_ir0 = [[inv] for inv in Inv]

elif instance_version == "smallsize2":
    for i in range(V):
        Inv.extend([
            df_V['Water StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)

#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print (Inv)
#print(g_shelter)
#print(g_pod)
#print(w_r)

In [12]:
# Define the model
m = gp.Model("Two-Stage Stochastic Programming Model")

# # Set the optimization direction
# m.modelSense = GRB.MINIMIZE

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [13]:
# Define decision variables 
# i, j is position, r is commodity, t is time, and s is scenario

# Define first-stage decision variables
# I: Inventory of commodity r at node i at the start of time period t
I = {}

for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = m.addVar(vtype=GRB.CONTINUOUS, name="I_%s_%s_%s" % (i, r, t))

# Define second-stage decision variables
U = {}  # Demand shortage at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set (e.g., 0 and 1)
            U[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="U_%s_%s_%s" % (i, t, s))

S = {}  # Demand satisfied at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set
            S[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="S_%s_%s_%s" % (i, t, s))

f = {}  # Quantity of commodity r traveling on link (i, j) at time period t in scenario s

for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only define f_ijrt^s if (i, j) is a valid link
            for r in range(R):
                for t in range(T):
                    for s in sce:  # Iterate directly over the scenario set
                        f[i, j, r, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="f_%s_%s_%s_%s_%s" % (i, j, r, t, s))
m.update()

In [14]:
# Define constraints
"""
# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            m.addConstr(
                I[i, r, t] <= I[i, r, t-1],
                name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            m.addConstr(
                I[i, r, t] >= I[i, r, t-1],
                name=f"inventory_increase_pod_{i}_{r}_{t}")

"""

#Starting Value of I[t=0]
for i in range(V):
    for r in range(R):
        m.addConstr(I[i,r,0]==I_ir0[i][r])

# Update demand shortage for all locations, time periods (except t=0), and scenarios
for i in range(V):
    for t in range(1, T):
        for s in sce:
            m.addConstr(U[i, t, s] == U[i, t-1, s] - S[i, t, s] + d_its[(i, t, s)], 
                        name="DemandShortageUpdate_%s_%s_%s" % (i, t, s))

# Initial demand shortage for all locations and scenarios at time period t=0
for i in range(V):
    for s in sce:
        m.addConstr(U[i, 0, s] == d_its[(i, 0, s)], name="InitialDemandShortage_%s_%s" % (i, s))


# Demand satisfaction constraint for all locations, time periods, and scenarios
for i in range(V):
    for t in range(T):
        for s in sce:
            m.addConstr(S[i, t, s] <= U[i, t, s], name="DemandSatisfaction_%s_%s_%s" % (i, t, s))

# Inventory update constraint for all locations, commodities, time periods (except last), and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                incoming_flow = gp.quicksum(f[j, i, r, t - t_ijt[j][i] + 1, s] for j in range(V) if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                
                # Inventory update constraint for shelters
                if i in V1:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_shelter[r],
                        name="InventoryUpdate_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for pods
                elif i in V3:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_pod[r],
                        name="InventoryUpdate_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for warehouses and RSAs
                else:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow,
                        name="InventoryUpdate_%s_%s_%s_%s" % (i, r, t, s)
                    )

# Inventory sufficiency constraint
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                if i in V1:  # Shelter
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_shelter[r],
                        name="InventorySufficiency_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                elif i in V3:  # Pod
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_pod[r],
                        name="InventorySufficiency_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                else:  # Warehouse or RSA
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow,
                        name="InventorySufficiency_%s_%s_%s_%s" % (i, r, t, s)
                    )


# Flow capacity constraint for all links, time periods, and scenarios
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in sce:
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        m.addConstr(
                            gp.quicksum(w_r[r] * f[i, j, r, t, s] for r in range(R)) <= K_ijt[i][j][t],
                            name="FlowCapacity_%s_%s_%s_%s" % (i, j, t, s)
                        )
m.update()

In [15]:
# Set up the objective function
objective = gp.quicksum(
    P[s] * gp.quicksum(
        gp.quicksum(
            p_i[i] * U[i, t, s]
            for i in range(V)
        ) for t in range(T)
    )
    for s in sce  # Iterate over the defined scenario indices
)
m.update()

m.setObjective(objective, GRB.MINIMIZE)
m.update()

In [16]:
# Optimize the model
m.optimize()

# Check if the optimization was successful
if m.status == GRB.OPTIMAL:
    # Create an empty list to store variable values
    variable_data = []

    # Loop through decision variables and get their values
    for var in m.getVars():
        variable_data.append([var.VarName, var.X])  # Append variable name and value

    # Create a DataFrame from the collected data
    df = pd.DataFrame(variable_data, columns=['Variable', 'Value'])

    # Save the DataFrame to an Excel file
    output_file = 'decision_variables_output_small.xlsx'
    df.to_excel(output_file, index=False, engine='openpyxl')

    print(f'Decision variable values have been saved to {output_file}')
else:
    print("The model did not find an optimal solution.")

Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 1317 rows, 981 columns and 3407 nonzeros
Model fingerprint: 0xefd867c4
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [3e-02, 4e-02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+03]
Presolve removed 1175 rows and 751 columns
Presolve time: 0.01s
Presolved: 142 rows, 230 columns, 819 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.1054015e+01   1.970037e+03   0.000000e+00      0s
      39    6.5115121e+02   0.000000e+00   0.000000e+00      0s

Solved in 39 iterations and 0.02 seconds (0.00 work units)
Optimal objective  6.511512149e+02
Decision variable values have been saved to decision_variables_output_small.xlsx
